# ESM vs feature association tests

DevSet1010 only. Does not retrain predictors.

- **Probe R²:** protein-grouped 5-fold ridge regression of each feature column on raw ESM. High out-of-fold R² means the column is linearly redundant with ESM.
- **Nested LRT:** conditional logistic regression stratified by protein (protein intercepts conditioned out; the fixed-effects form of a protein-level GLMM). Compare ESM-only vs ESM + feature block. A significant likelihood-ratio test means the block still tracks binding after ESM.

ESM enters the LRT as 32 unsupervised principal components. A block with dimension ≤ 6 enters as its standardized columns; a wider block enters as its first 6 principal components.

Verdicts use mean Probe R² ≥ 0.50 as high overlap and Benjamini–Hochberg FDR < 0.05 as a significant LRT:

- low R² + significant LRT: complementary to ESM and associated with binding
- high R² + non-significant LRT: redundant with ESM
- high R² + significant LRT: overlaps ESM but still has leftover binding signal
- low R² + non-significant LRT: not a copy of ESM, and not associated with binding after ESM


In [ ]:
# ============================================================
# 0. DRIVE / INSTALL / IMPORT
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, subprocess, warnings

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels'],
    check=True,
)

import numpy as np
import torch
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.discrete.conditional_models import ConditionalLogit

warnings.filterwarnings('ignore', category=FutureWarning)

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
ULTIMATE_CANDIDATES = [
    BASE / 'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE / 'GEAR_ULTIMATE',
]
ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p / 'gear_ultimate').exists()),
    None,
)
if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE.'
    )

req = ULTIMATE_DIR / 'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)],
        check=True,
    )

sys.path.insert(0, str(ULTIMATE_DIR))
from gear_ultimate import gear_suite as suite

OUT_DIR = BASE / 'results' / 'esm_feature_association'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ESM_BATCH_SIZE = 8
SEED = 2026
N_ESM_PCS = 32
BLOCK_PC_CAP = 6
PROBE_SPLITS = 5
PROBE_ALPHAS = np.logspace(-1, 5, 8)
HIGH_R2 = 0.50
FDR_ALPHA = 0.05

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# ============================================================
# 1. DEVSET LABELS
# ============================================================
import ast

TRAIN_CSV = BASE / 'data' / 'development_set' / 'full_grouped_train_binding_sites_df.csv'

def parse_list(value):
    if isinstance(value, str):
        return list(ast.literal_eval(value))
    return list(value)

LABEL_COLUMNS = [
    'metal_binding',
    'nuclear_binding',
    'small_binding',
    'any_ligand_binding_sites',
]

dev_df = pd.read_csv(TRAIN_CSV)
dev_df['prot_id'] = dev_df['prot_id'].astype(str)
dev_df['sequence'] = dev_df['sequence'].astype(str)
for column in LABEL_COLUMNS:
    if column in dev_df.columns:
        dev_df[column] = dev_df[column].apply(parse_list)

# Official DevSet1010 order, same source the ligand notebooks use.
fold_root_candidates = [
    BASE / 'baseline_reproduction_master_v6' / 'repos' / 'bindnode24' / 'dataset' / 'cv_splits' / 'predicted_structures',
    BASE / 'repos' / 'bindnode24' / 'dataset' / 'cv_splits' / 'predicted_structures',
]
fold_root = next((p for p in fold_root_candidates if p.exists()), None)
if fold_root is None:
    raise FileNotFoundError('Official cv split files were not found on Drive.')

official_order = []
seen = set()
for fold_id in range(1, 6):
    ids = [
        line.strip()
        for line in (fold_root / f'ids_split{fold_id}.txt').read_text().splitlines()
        if line.strip()
    ]
    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {fold_id}')
    official_order.extend(ids)
    seen.update(ids)

dev_df = (
    dev_df.set_index('prot_id', drop=False)
    .loc[official_order]
    .reset_index(drop=True)
)
assert len(dev_df) == 1010
print('DevSet1010:', len(dev_df), 'proteins')


In [ ]:
# ============================================================
# 2. STORE HELPERS + STATISTICS
# ============================================================
SHARED_BLOCKS = [
    'AA', 'SPATIAL', 'LOCAL', 'ATOM', 'CHEM', 'ACCESS', 'SHELL2',
    'ENV', 'CONF', 'METALSPEC', 'ELEC3D', 'TOPO', 'SOLV',
    'SEQMOTIF', 'SEQCTX',
]

def get_record(store, prot_id):
    prot_id = str(prot_id)
    if prot_id in store:
        return store[prot_id]
    raise KeyError(prot_id)

def get_block(store, prot_id, block):
    record = get_record(store, prot_id)
    if isinstance(record, dict):
        if block in record:
            return np.asarray(record[block])
        features = record.get('features')
        if isinstance(features, dict) and block in features:
            return np.asarray(features[block])
    if hasattr(record, block):
        return np.asarray(getattr(record, block))
    for attr in ('features', 'feature_dict', 'blocks'):
        mapping = getattr(record, attr, None)
        if isinstance(mapping, dict) and block in mapping:
            return np.asarray(mapping[block])
    raise KeyError(f'{prot_id}/{block}')

def set_block(store, prot_id, block, values):
    record = get_record(store, prot_id)
    values = np.asarray(values, dtype=np.float32)
    if isinstance(record, dict):
        features = record.get('features')
        if isinstance(features, dict):
            features[block] = values
        else:
            record[block] = values
        return
    for attr in ('features', 'feature_dict', 'blocks'):
        mapping = getattr(record, attr, None)
        if isinstance(mapping, dict):
            mapping[block] = values
            return
    setattr(record, block, values)

def list_store_blocks(store, prot_id):
    record = get_record(store, prot_id)
    keys = []
    if isinstance(record, dict):
        features = record.get('features')
        keys = list(features) if isinstance(features, dict) else list(record)
    else:
        for attr in ('features', 'feature_dict', 'blocks'):
            mapping = getattr(record, attr, None)
            if isinstance(mapping, dict):
                keys = list(mapping)
                break
    return [key for key in keys if key != 'ESM' and not str(key).startswith('_')]

def attach_npz_blocks(store, frame, cache_dir, expected):
    """Attach cached ligand blocks when every DevSet protein file is present."""
    cache_dir = Path(cache_dir)
    paths = [cache_dir / f'{prot_id}.npz' for prot_id in frame['prot_id']]
    if not all(path.exists() for path in paths):
        missing = sum(not path.exists() for path in paths)
        print(f'Skip cache {cache_dir}: missing {missing}/{len(paths)} files')
        return []
    attached = []
    for row, path in zip(frame.itertuples(), paths):
        archive = np.load(path)
        for name in expected:
            values = np.asarray(archive[name], dtype=np.float32)
            if values.shape[0] != len(row.sequence):
                raise RuntimeError(
                    f'{row.prot_id}/{name}: {values.shape} vs L={len(row.sequence)}'
                )
            set_block(store, row.prot_id, name, values)
        attached = list(expected)
    print('Attached', attached, 'from', cache_dir)
    return attached

def complete_blocks(store, frame, block_names):
    """Keep blocks present at the sequence length for every protein."""
    kept = []
    for name in block_names:
        ok = True
        for row in frame.itertuples():
            try:
                values = get_block(store, row.prot_id, name)
            except KeyError:
                ok = False
                break
            values = np.asarray(values)
            if values.ndim != 2 or values.shape[0] != len(row.sequence):
                ok = False
                break
        if ok:
            kept.append(name)
        else:
            print(f'Skip incomplete block {name}')
    return kept

def stack_residue_table(store, frame, label_column, block_names):
    """Return ESM, labels, protein ids, and a dict of feature matrices."""
    block_names = complete_blocks(store, frame, block_names)
    esm_parts = []
    y_parts = []
    id_parts = []
    block_parts = {name: [] for name in block_names}
    for row in frame.itertuples():
        length = len(row.sequence)
        labels = np.asarray(getattr(row, label_column), dtype=np.float64)
        embedding = np.asarray(get_block(store, row.prot_id, 'ESM'), dtype=np.float32)
        if embedding.shape[0] != length or len(labels) != length:
            raise RuntimeError(f'{row.prot_id}: ESM/label length mismatch')
        esm_parts.append(embedding)
        y_parts.append(labels)
        id_parts.append(np.full(length, row.prot_id, dtype=object))
        for name in block_names:
            block_parts[name].append(
                np.asarray(get_block(store, row.prot_id, name), dtype=np.float32)
            )
    return {
        'esm': np.concatenate(esm_parts, axis=0),
        'y': np.concatenate(y_parts, axis=0),
        'prot_id': np.concatenate(id_parts, axis=0),
        'blocks': {
            name: np.concatenate(parts, axis=0)
            for name, parts in block_parts.items()
        },
        'n_proteins': int(len(frame)),
    }

def _ridge_weights(x_train, y_train, alpha):
    gram = x_train.T @ x_train
    gram = gram.astype(np.float64, copy=False)
    gram.flat[:: gram.shape[0] + 1] += float(alpha)
    rhs = x_train.T @ y_train
    return np.linalg.solve(gram, rhs.astype(np.float64, copy=False))

def probe_r2_block(esm, features, groups):
    """Out-of-fold R² of each feature column predicted from raw ESM."""
    n_cols = features.shape[1]
    oof = np.full((features.shape[0], n_cols), np.nan, dtype=np.float64)
    splitter = GroupKFold(n_splits=min(PROBE_SPLITS, len(np.unique(groups))))
    for train_idx, test_idx in splitter.split(esm, groups=groups):
        inner = GroupKFold(n_splits=min(5, len(np.unique(groups[train_idx]))))
        inner_train, inner_val = next(inner.split(esm[train_idx], groups=groups[train_idx]))
        fit_idx = train_idx[inner_train]
        val_idx = train_idx[inner_val]
        scaler = StandardScaler()
        x_fit = scaler.fit_transform(esm[fit_idx]).astype(np.float32, copy=False)
        x_val = scaler.transform(esm[val_idx]).astype(np.float32, copy=False)
        y_mu = features[fit_idx].mean(axis=0)
        y_fit = features[fit_idx] - y_mu
        mse = []
        for alpha in PROBE_ALPHAS:
            weight = _ridge_weights(x_fit, y_fit, alpha)
            pred = x_val @ weight.astype(np.float32) + y_mu
            mse.append(np.mean((features[val_idx] - pred) ** 2, axis=0))
        mse = np.vstack(mse)
        chosen = PROBE_ALPHAS[np.argmin(mse, axis=0)]
        scaler = StandardScaler()
        x_train = scaler.fit_transform(esm[train_idx]).astype(np.float32, copy=False)
        x_test = scaler.transform(esm[test_idx]).astype(np.float32, copy=False)
        y_mu = features[train_idx].mean(axis=0)
        y_train = features[train_idx] - y_mu
        prediction = np.empty((len(test_idx), n_cols), dtype=np.float64)
        for alpha in np.unique(chosen):
            cols = np.flatnonzero(chosen == alpha)
            weight = _ridge_weights(x_train, y_train[:, cols], alpha)
            prediction[:, cols] = x_test @ weight.astype(np.float32) + y_mu[cols]
        oof[test_idx] = prediction
    y_mu = features.mean(axis=0)
    ss_tot = np.sum((features - y_mu) ** 2, axis=0)
    ss_res = np.sum((features - oof) ** 2, axis=0)
    r2 = np.full(n_cols, np.nan, dtype=np.float64)
    usable = ss_tot > 1e-8
    r2[usable] = 1.0 - ss_res[usable] / ss_tot[usable]
    return r2

def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(p_values.shape, np.nan)
    valid = np.isfinite(p_values)
    raw = p_values[valid]
    m = len(raw)
    if m == 0:
        return adjusted
    order = np.argsort(raw)
    ranked = raw[order] * m / np.arange(1, m + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    ranked = np.clip(ranked, 0.0, 1.0)
    out = np.empty(m, dtype=float)
    out[order] = ranked
    adjusted[valid] = out
    return adjusted

def _varying_mask(y, groups):
    keep = np.zeros(len(y), dtype=bool)
    for group in pd.unique(groups):
        idx = np.flatnonzero(groups == group)
        labels = y[idx]
        if len(labels) >= 2 and labels.min() != labels.max():
            keep[idx] = True
    return keep

def _design_pcs(values, n_components, unsupervised_columns):
    """Standardize. Use raw columns when dim <= cap, else PCA."""
    scaler = StandardScaler()
    scaled = scaler.fit_transform(values)
    dim = scaled.shape[1]
    if dim <= unsupervised_columns:
        keep = np.nanstd(scaled, axis=0) > 1e-8
        if not np.any(keep):
            raise RuntimeError('Block has no variance')
        return scaled[:, keep]
    n_components = min(n_components, dim, scaled.shape[0] - 1)
    pca = PCA(
        n_components=n_components,
        svd_solver='randomized',
        random_state=SEED,
    )
    return pca.fit_transform(scaled)

def nested_lrt(y, groups, esm, block):
    """Protein-stratified conditional logit: ESM PCs vs ESM PCs + block."""
    mask = _varying_mask(y, groups)
    y = y[mask]
    groups = groups[mask]
    esm = esm[mask]
    block = block[mask]
    esm_design = _design_pcs(esm, N_ESM_PCS, N_ESM_PCS)
    block_design = _design_pcs(block, BLOCK_PC_CAP, BLOCK_PC_CAP)
    order = np.argsort(groups, kind='mergesort')
    y = y[order]
    groups = groups[order]
    esm_design = esm_design[order]
    block_design = block_design[order]
    codes = pd.factorize(groups, sort=False)[0]
    null = ConditionalLogit(y, esm_design, groups=codes).fit(
        disp=0, method='newton', maxiter=100,
    )
    alt_design = np.concatenate([esm_design, block_design], axis=1)
    alt = ConditionalLogit(y, alt_design, groups=codes).fit(
        disp=0, method='newton', maxiter=100,
    )
    df = int(block_design.shape[1])
    lr_stat = float(2.0 * (alt.llf - null.llf))
    if lr_stat < 0:
        lr_stat = np.nan
        p_value = np.nan
    else:
        p_value = float(stats.chi2.sf(lr_stat, df))
    return {
        'n_residues': int(len(y)),
        'n_proteins': int(len(np.unique(codes))),
        'n_esm_pcs': int(esm_design.shape[1]),
        'n_block_pcs': df,
        'll_m0': float(null.llf),
        'll_m1': float(alt.llf),
        'lr_stat': lr_stat,
        'df': df,
        'p_value': p_value,
    }

def verdict_row(mean_r2, fdr):
    high = np.isfinite(mean_r2) and mean_r2 >= HIGH_R2
    sig = np.isfinite(fdr) and fdr < FDR_ALPHA
    if high and sig:
        return 'overlaps_esm_residual_binding_signal'
    if high and not sig:
        return 'redundant_with_esm'
    if (not high) and sig:
        return 'complementary_and_associated'
    return 'not_redundant_not_associated'

print('Helpers ready')


In [ ]:
# ============================================================
# 3. LIGAND CONFIGS
# ============================================================
LIGANDS = {
    'small': {
        'label_column': 'small_binding',
        'run_version': 'small_ultimate_v1_context',
        'models': ('MLP',),
        'cache_dir': BASE / 'cache' / 'small_ultimate_v1' / 'small_explicit_v1' / 'DevSet1010',
        'extra_blocks': [
            'SM_SEQCHEM32', 'SM_POCKET3D44', 'SM_LIGCHEM20',
            'SM_TOPO16', 'SM_AROMSTACK12', 'SM_PROTCTX8',
        ],
    },
    'nuclear': {
        'label_column': 'nuclear_binding',
        'run_version': 'nuclear_ultimate_v1_context',
        'models': ('MLP',),
        'cache_dir': BASE / 'cache' / 'nuclear_ultimate_v1' / 'nuclear_explicit_v1' / 'DevSet1010',
        'extra_blocks': [
            'NA_SEQELECTRO32', 'NA_PATCH3D40', 'NA_HBOND16',
            'NA_TOPO12', 'NA_PROTCTX8', 'NA_AROMCTRL8',
        ],
    },
    'metal': {
        'label_column': 'metal_binding',
        'run_version': 'metal_v2_context',
        'models': ('MLP', 'KAN', 'XGBoost'),
        'cache_dir': None,
        'extra_blocks': [
            'METALSPEC', 'CHEM', 'TOPO', 'ACCESS', 'ATOM',
            'SOLV', 'SPATIAL', 'LOCAL', 'SEQMOTIF',
        ],
    },
}

RUN_LIGANDS = ['small', 'nuclear', 'metal']
print('Will run:', RUN_LIGANDS)


In [ ]:
# ============================================================
# 4. RUN PROBE R² + NESTED LRT
# ============================================================
summaries = []

for ligand in RUN_LIGANDS:
    cfg = LIGANDS[ligand]
    print('=' * 72)
    print('LIGAND', ligand)
    ctx = suite.prepare_context(
        base=BASE,
        run_version=cfg['run_version'],
        label_column=cfg['label_column'],
        models=cfg['models'],
        ensure_base=True,
        add_mech=True,
        add_multi=True,
        add_seqctx=True,
        esm_batch_size=ESM_BATCH_SIZE,
        device=torch.device('cpu'),
    )
    store = ctx.stores['train']
    if len(store) != 1010:
        raise RuntimeError(f'{ligand}: expected 1010 train proteins, got {len(store)}')

    if cfg['cache_dir'] is not None:
        attach_npz_blocks(store, dev_df, cfg['cache_dir'], cfg['extra_blocks'])

    discovered = list_store_blocks(store, dev_df.iloc[0]['prot_id'])
    block_names = []
    for name in list(cfg['extra_blocks']) + SHARED_BLOCKS + discovered:
        if name not in block_names and name != 'ESM':
            block_names.append(name)

    table = stack_residue_table(
        store, dev_df, cfg['label_column'], block_names,
    )
    present = [name for name in block_names if name in table['blocks']]
    print(
        f"stacked proteins={table['n_proteins']} residues={len(table['y'])} "
        f"positives={int(table['y'].sum())} blocks={present}"
    )

    probe_rows = []
    lrt_rows = []
    for name in present:
        features = table['blocks'][name].astype(np.float32, copy=False)
        print(f'  probe {name} dim={features.shape[1]}')
        r2 = probe_r2_block(table['esm'], features, table['prot_id'])
        for index, value in enumerate(r2):
            probe_rows.append({
                'ligand': ligand,
                'block': name,
                'feature_index': index,
                'oof_r2': value,
                'n_residues': int(len(table['y'])),
                'n_proteins': int(table['n_proteins']),
            })
        print(f'  lrt {name}')
        try:
            stats_row = nested_lrt(
                table['y'], table['prot_id'], table['esm'], features,
            )
        except Exception as exc:
            print('  LRT failed', name, repr(exc))
            stats_row = {
                'n_residues': np.nan,
                'n_proteins': np.nan,
                'n_esm_pcs': np.nan,
                'n_block_pcs': np.nan,
                'll_m0': np.nan,
                'll_m1': np.nan,
                'lr_stat': np.nan,
                'df': np.nan,
                'p_value': np.nan,
            }
        stats_row.update({'ligand': ligand, 'block': name, 'block_dim': int(features.shape[1])})
        lrt_rows.append(stats_row)

    probe_df = pd.DataFrame(probe_rows)
    lrt_df = pd.DataFrame(lrt_rows)
    lrt_df['fdr'] = benjamini_hochberg(lrt_df['p_value'].to_numpy())
    probe_path = OUT_DIR / f'esm_probe_r2_{ligand}.csv'
    lrt_path = OUT_DIR / f'esm_nested_lrt_{ligand}.csv'
    probe_df.to_csv(probe_path, index=False)
    lrt_df.to_csv(lrt_path, index=False)

    summary = (
        probe_df.groupby('block', sort=False)['oof_r2']
        .agg(probe_r2_mean='mean', probe_r2_median='median')
        .reset_index()
    )
    summary = summary.merge(lrt_df, on='block', how='outer')
    summary['verdict'] = [
        verdict_row(mean_r2, fdr)
        for mean_r2, fdr in zip(summary['probe_r2_mean'], summary['fdr'])
    ]
    verdict_path = OUT_DIR / f'esm_feature_verdicts_{ligand}.csv'
    summary.to_csv(verdict_path, index=False)
    summaries.append(summary)
    print('Wrote', probe_path)
    print('Wrote', lrt_path)
    print('Wrote', verdict_path)
    display(summary)

print('Finished ligands:', RUN_LIGANDS)


In [ ]:
# ============================================================
# 5. MARKDOWN SUMMARY
# ============================================================
lines = [
    '# ESM feature association (DevSet1010)',
    '',
    f'High Probe R² means mean out-of-fold R² ≥ {HIGH_R2:.2f}.',
    f'Significant LRT means BH FDR < {FDR_ALPHA:.2f}.',
    '',
]
for summary in summaries:
    ligand = summary['ligand'].iloc[0]
    lines.append(f'## {ligand}')
    lines.append('')
    lines.append('| block | probe R² mean | probe R² median | LRT | df | FDR | verdict |')
    lines.append('|---|---:|---:|---:|---:|---:|---|')
    for row in summary.itertuples(index=False):
        lines.append(
            '| {block} | {mean:.3f} | {med:.3f} | {lr} | {df} | {fdr} | {verdict} |'.format(
                block=row.block,
                mean=row.probe_r2_mean,
                med=row.probe_r2_median,
                lr='nan' if not np.isfinite(row.lr_stat) else f'{row.lr_stat:.2f}',
                df='nan' if not np.isfinite(row.df) else str(int(row.df)),
                fdr='nan' if not np.isfinite(row.fdr) else f'{row.fdr:.3g}',
                verdict=row.verdict,
            )
        )
    lines.append('')

report = '\n'.join(lines)
print(report)
report_path = OUT_DIR / 'esm_feature_summary.md'
report_path.write_text(report)
print('Wrote', report_path)
